# Neural validator - version 2 
Trains a neural network to predict bad moves for the baseline agent in the Orbit Wars environment.

## Dataset & Env setup

In [ ]:
from __future__ import annotations

import csv
import glob
import importlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import tarfile
import time
from pathlib import Path
from typing import Any
import numpy as np

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUT_DIR = WORK / 'nn_validator_submission_c'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('input exists:', KAGGLE_INPUT.exists())
print('out:', OUT_DIR)


input exists: True
out: /kaggle/working/nn_validator_submission_c


In [ ]:
def _has_orbit_wars_env() -> bool:
    try:
        import kaggle_environments
    except ModuleNotFoundError:
        return False
    env_root = Path(kaggle_environments.__file__).resolve().parent / 'envs'
    return (env_root / 'orbit_wars').exists()

if not _has_orbit_wars_env():
    wheels = sorted(KAGGLE_INPUT.rglob('kaggle_environments-*.whl'))
    assert wheels, 'orbit_wars env not found and no kaggle_environments-*.whl found under /kaggle/input'
    wheel = wheels[0]
    print('Installing local wheel:', wheel)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', str(wheel)], check=True)
    importlib.invalidate_caches()
    if 'kaggle_environments' in sys.modules:
        importlib.reload(sys.modules['kaggle_environments'])

from kaggle_environments import make
print('orbit_wars available:', _has_orbit_wars_env())
assert _has_orbit_wars_env(), 'orbit_wars environment still unavailable after setup'


orbit_wars available: True


## Baseline & Opponents

In [ ]:
def find_submission(name: str, root: Path) -> Path | None:
    for main_file in root.rglob("main.py"):
        if main_file.parent.name == name:
            return main_file.parent
    return None

def pick_closest(paths: list[Path]) -> Path:
    return sorted(paths, key=lambda p: (len(p.parts), str(p)))[0]

teacher_dir = find_submission("submission_c", KAGGLE_INPUT)
assert teacher_dir, "Missing submission_c/main.py in dataset"
teacher_path = teacher_dir / "main.py"
peer_h_dir = find_submission("submission_h", KAGGLE_INPUT)
peer_h_path = peer_h_dir / "main.py" if peer_h_dir else None
opponent_names = [
    "buddy.py",
    "multi_focus_hybrid_v1a_enemy_prod_bias_v2.py",
    "ver12.py",
]

include_peer_h = True
include_selfplay = True
all_py_files = list(KAGGLE_INPUT.rglob("*.py"))
opponents = []
if include_peer_h and peer_h_path:
    opponents.append(peer_h_path)
candidates_by_name = {}
for p in all_py_files:
    rp = p.resolve()

    if rp == teacher_path.resolve():
        continue
    if peer_h_path and rp == peer_h_path.resolve():
        continue

    if p.name in opponent_names:
        candidates_by_name.setdefault(p.name, []).append(p)
missing = []
for name in opponent_names:
    matches = candidates_by_name.get(name, [])
    if not matches:
        missing.append(name)
        continue
    opponents.append(pick_closest(matches))
if include_selfplay:
    opponents.append(teacher_path)

seen = set()
final_opponents = []

for p in opponents:
    key = str(p.resolve())
    if key in seen:
        continue
    seen.add(key)
    final_opponents.append(p)

print("teacher:", teacher_path)
print("peer_h:", peer_h_path or "(not found)")
print("opponent list:", opponent_names)

if missing:
    raise AssertionError(f"Missing opponents: {missing}")

print("total opponents:", len(final_opponents))
for p in final_opponents:
    print(" -", p)

assert final_opponents, "No opponents found"

teacher: /kaggle/input/datasets/thanhhuyen197abs/submission-c/submission_c/main.py
peer_h: /kaggle/input/datasets/thanhhuyen197abs/submission-h/submission_h/main.py
selected opponent basenames: ['buddy.py', 'multi_focus_hybrid_v1a_enemy_prod_bias_v2.py', 'ver12.py']
opponents used: 5
  /kaggle/input/datasets/thanhhuyen197abs/submission-h/submission_h/main.py
  /kaggle/input/datasets/cunmincut/opponents/buddy.py
  /kaggle/input/datasets/cunmincut/opponents/multi_focus_hybrid_v1a_enemy_prod_bias_v2.py
  /kaggle/input/datasets/cunmincut/opponents/ver12.py
  /kaggle/input/datasets/thanhhuyen197abs/submission-c/submission_c/main.py


## Config

In [ ]:
TRAIN_SCALE = 'full'  # 'smoke', 'small', or 'full'

if TRAIN_SCALE == 'smoke':
    SEEDS = list(range(101, 106))          
    SELFPLAY_SEEDS = list(range(201, 206)) 
elif TRAIN_SCALE == 'small':
    SEEDS = list(range(101, 111))          
    SELFPLAY_SEEDS = list(range(201, 211)) 
elif TRAIN_SCALE == 'full':
    SEEDS = list(range(101, 121))          
    SELFPLAY_SEEDS = list(range(201, 221)) 
else:
    raise ValueError(f'Unknown TRAIN_SCALE: {TRAIN_SCALE}')

non_self_count = sum(1 for p in OPPONENT_PATHS if p.resolve() != TEACHER_PATH.resolve())
self_count = sum(1 for p in OPPONENT_PATHS if p.resolve() == TEACHER_PATH.resolve())
estimated_games = non_self_count * len(SEEDS) * 2 + self_count * len(SELFPLAY_SEEDS) * 2

print('train_scale:', TRAIN_SCALE)
print('seeds:', SEEDS)
print('selfplay_seeds:', SELFPLAY_SEEDS)
print('active opponents:', len(OPPONENT_PATHS))
print('estimated games:', estimated_games)


train_scale: full
seeds: [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120]
selfplay_seeds: [201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220]
active opponents: 5
estimated games: 200


## Baseline model & Features

In [ ]:
if str(TEACHER_DIR) not in sys.path:
    sys.path.insert(0, str(TEACHER_DIR))

spec = importlib.util.spec_from_file_location('submission_c_teacher_for_nn_validator', TEACHER_PATH)
C_MOD = importlib.util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
sys.modules[spec.name] = C_MOD
spec.loader.exec_module(C_MOD)

import torch
from orbit_lite.adapter import single_obs_to_tensor
from orbit_lite.obs import parse_obs
from orbit_lite.distance_cache import build_distance_cache
from orbit_lite.movement_step import ensure_planet_movement
from orbit_lite.planner_core import largest_initial_player_count

FEATURE_NAMES = [
    'score_base', 'score_margin', 'eta_norm', 'ships_sent_norm', 'send_ratio',
    'source_ships_norm', 'send_proxy_norm', 'source_after_norm',
    'target_ships_norm', 'target_prod_norm',
    'target_neutral', 'target_enemy', 'target_mine',
    'approx_capture_margin_norm', 'approx_capture_ratio',
    'step_norm', 'is_4p', 'player_count_norm',
    'my_prod_share', 'my_ship_share', 'prod_lead_norm', 'ship_lead_norm',
    'leader_delta_norm', 'target_owner_strength_norm',
    'pressure_src_norm', 'pressure_tgt_norm', 'pressure_gap_norm',
    'enemy_fleet_count_norm', 'enemy_fleet_ships_norm',
    'target_prod_bonus_proxy',
]
FEATURE_DIM = len(FEATURE_NAMES)
print('feature_dim:', FEATURE_DIM)


feature_dim: 20


## Labeling

In [ ]:
def _angle_delta(a: float, b: float) -> float:
    return abs(math.atan2(math.sin(a - b), math.cos(a - b)))

def _owner_at(env_steps, slot: int, turn: int, planet_id: int):
    if turn < 0 or turn >= len(env_steps):
        return None
    obs = env_steps[turn][slot].observation
    if obs is None:
        return None
    for p in obs.get('planets', []):
        if int(p[0]) == int(planet_id):
            return int(p[1])
    return None

def _first_owned_turn(env_steps, slot: int, planet_id: int, player_id: int, start: int, end: int):
    end = min(end, len(env_steps) - 1)
    for t in range(max(0, start), end + 1):
        if _owner_at(env_steps, slot, t, planet_id) == int(player_id):
            return t
    return None

def label_bad_move(env_steps, *, slot: int, player_id: int, source_id: int, target_id: int, step_idx: int, eta: float):
    arrival = step_idx + max(1, int(math.ceil(float(eta))))
    first_owned = _first_owned_turn(env_steps, slot, target_id, player_id, arrival, arrival + 8)
    captured = first_owned is not None

    source_lost = False
    for t in range(step_idx + 1, min(step_idx + 19, len(env_steps))):
        owner = _owner_at(env_steps, slot, t, source_id)
        if owner is not None and owner != int(player_id):
            source_lost = True
            break

    target_lost_quick = False
    if first_owned is not None:
        for t in range(first_owned + 1, min(first_owned + 9, len(env_steps))):
            owner = _owner_at(env_steps, slot, t, target_id)
            if owner is not None and owner != int(player_id):
                target_lost_quick = True
                break

    bad = (not captured) or source_lost or target_lost_quick
    return int(bad), {
        'arrival_turn': arrival,
        'captured': int(captured),
        'source_lost': int(source_lost),
        'target_lost_quick': int(target_lost_quick),
    }

def match_action_to_candidate(action, cand_rows):
    if not cand_rows:
        return None
    sid, ang, ships = int(action[0]), float(action[1]), int(action[2])
    best = None
    best_key = None
    for row in cand_rows:
        if row['source_id'] != sid:
            continue
        ship_gap = abs(int(row['ships']) - ships)
        if ship_gap > 1:
            continue
        da = _angle_delta(float(row['angle']), ang)
        key = (ship_gap, da)
        if best is None or key < best_key:
            best = row
            best_key = key
    if best is not None and best_key[1] < 0.05:
        return best
    return None

def agent_label_for_path(path: Path):
    rp = Path(path).resolve()
    if PEER_H_PATH is not None and rp == PEER_H_PATH.resolve():
        return 'submission_h'
    if rp == TEACHER_PATH.resolve():
        return 'submission_c_self'
    return rp.stem


## Greedy move feature extraction

In [ ]:
def rows_from_greedy_capture(obs_dict: dict[str, Any], player_id: int, cap: dict[str, Any]):
    obs_tensors = single_obs_to_tensor(obs_dict, player_id=int(player_id), device='cpu')
    obs = parse_obs(obs_tensors)
    P = int(obs.P)
    if P == 0:
        return []
    player_count = largest_initial_player_count(obs_tensors)
    config = C_MOD._config_for(player_count)
    planet_ids = obs_tensors['planets'][..., 0].long()
    prod = obs_tensors['planets'][..., 6].to(torch.float32)

    try:
        movement = ensure_planet_movement(
            obs_tensors=obs_tensors,
            expected_cfg=C_MOD._movement_config(config, player_count=int(player_count)),
            cached_movement=None,
        )
        cache = build_distance_cache(movement, max_k=int(config.horizon))
        pressure = C_MOD.cheap_enemy_pressure(obs, cache, horizon=float(config.horizon), player_id=int(player_id))
        prod = movement.planet_prod.to(torch.float32)
    except Exception:
        pressure = torch.zeros(P, dtype=torch.float32)

    score = cap['score'].to(torch.float32)
    cand_src = cap['cand_src'].to(torch.long)
    cand_send = cap['cand_send'].to(torch.float32)
    cand_angle = cap['cand_angle'].to(torch.float32)
    cand_eta = cap['cand_eta'].to(torch.float32)
    cand_active = cap['cand_active'].to(torch.bool)
    cand_tgt_slot = cap['cand_tgt_slot'].to(torch.long)
    cand_is_def = cap['cand_is_def'].to(torch.bool)
    roi_threshold = float(cap['roi_threshold'])

    alive = obs.alive
    owned = obs.owned & alive
    enemy_owned = obs.is_enemy & alive
    total_prod = prod[alive].sum().clamp(min=1e-6)
    total_owned_ships = obs.ships[(obs.owner_abs >= 0) & alive].sum().clamp(min=1e-6)
    my_prod = prod[owned].sum()
    enemy_prod = prod[enemy_owned].sum()
    my_ships = obs.ships[owned].sum()
    enemy_ships = obs.ships[enemy_owned].sum()
    my_prod_share = my_prod / total_prod
    my_ship_share = my_ships / total_owned_ships
    prod_lead = (my_prod - enemy_prod) / 50.0
    ship_lead = (my_ships - enemy_ships) / 200.0

    pid = int(player_id)
    dtype = torch.float32
    leader_strength = torch.zeros(P, dtype=dtype)
    leader_delta_by_slot = torch.zeros(P, dtype=dtype)
    if int(player_count) >= 4:
        owner = obs.owner_abs.to(torch.long)
        owner_valid = (owner >= 0) & (owner < int(player_count)) & obs.alive
        owner_idx = owner.clamp(min=0, max=max(int(player_count) - 1, 0))
        prod_by_owner = torch.zeros(int(player_count), dtype=dtype)
        ships_by_owner = torch.zeros(int(player_count), dtype=dtype)
        prod_by_owner.scatter_add_(0, owner_idx, torch.where(owner_valid, prod.to(dtype), torch.zeros_like(prod.to(dtype))))
        ships_by_owner.scatter_add_(0, owner_idx, torch.where(owner_valid, obs.ships.to(dtype), torch.zeros_like(obs.ships.to(dtype))))
        strength = prod_by_owner + 0.025 * ships_by_owner
        my_strength = strength[pid].detach()
        safe_owner = owner.clamp(min=0, max=max(int(player_count) - 1, 0))
        leader_strength = strength[safe_owner]
        leader_delta_by_slot = (leader_strength - my_strength).clamp(min=0.0)

    f_enemy = obs.f_alive & (obs.f_owner.to(torch.long) >= 0) & (obs.f_owner.to(torch.long) != pid)
    enemy_fleet_count = f_enemy.to(dtype).sum() / 32.0
    enemy_fleet_ships = torch.where(f_enemy, obs.f_ships.to(dtype), torch.zeros_like(obs.f_ships.to(dtype))).sum() / 200.0

    rows = []
    C = int(score.shape[0])
    for ci in range(C):
        if cand_is_def[ci].item():
            continue
        if not torch.isfinite(score[ci]).item():
            continue
        if not bool(cand_active[ci].any().item()):
            continue
        src_slot = int(cand_src[ci, 0].item())
        tgt_slot = int(cand_tgt_slot[ci].item())
        src_safe = max(0, min(src_slot, P - 1))
        tgt_safe = max(0, min(tgt_slot, P - 1))
        src_id = int(planet_ids[src_safe].item())
        tgt_id = int(planet_ids[tgt_safe].item())
        if src_id < 0 or tgt_id < 0:
            continue
        send = float(cand_send[ci, 0].item())
        if send < 1.0:
            continue
        src_sh = float(obs.ships[src_safe].item())
        tgt_sh = float(obs.ships[tgt_safe].item())
        tgt_prod = float(prod[tgt_safe].item())
        src_after = max(src_sh - send, 0.0)
        eta = float(cand_eta[ci, 0].item())
        score_base = float(score[ci].item())
        p_src = float(pressure[src_safe].item())
        p_tgt = float(pressure[tgt_safe].item())
        owner = int(obs.owner_abs[tgt_safe].item())
        target_mine = 1.0 if owner == pid else 0.0
        target_neutral = 1.0 if owner < 0 else 0.0
        target_enemy = 1.0 if owner >= 0 and owner != pid else 0.0
        approx_floor = 1.0 if target_mine else max(tgt_sh + 1.0, 1.0)
        cap_margin = send - approx_floor
        cap_ratio = send / max(approx_floor, 1.0)
        leader_delta = float(leader_delta_by_slot[tgt_safe].item()) if target_enemy else 0.0
        owner_strength = float(leader_strength[tgt_safe].item()) if target_enemy else 0.0
        target_prod_bonus_proxy = tgt_prod if int(player_count) >= 4 and target_enemy else 0.0

        feat = np.array([
            score_base / 20.0,
            (score_base - roi_threshold) / 10.0,
            eta / 20.0,
            send / 100.0,
            send / max(src_sh, 1.0),
            src_sh / 100.0,
            send / 100.0,
            src_after / 100.0,
            tgt_sh / 100.0,
            tgt_prod / 5.0,
            target_neutral, target_enemy, target_mine,
            cap_margin / 50.0,
            cap_ratio / 3.0,
            float(int(obs_dict.get('step', 0))) / 500.0,
            1.0 if int(player_count) >= 4 else 0.0,
            float(player_count) / 4.0,
            float(my_prod_share.item()),
            float(my_ship_share.item()),
            float(prod_lead.item()),
            float(ship_lead.item()),
            leader_delta / 50.0,
            owner_strength / 50.0,
            p_src / 100.0,
            p_tgt / 100.0,
            (p_tgt - p_src) / 100.0,
            float(enemy_fleet_count.item()),
            float(enemy_fleet_ships.item()),
            target_prod_bonus_proxy / 5.0,
        ], dtype=np.float32)
        assert feat.shape[0] == FEATURE_DIM
        rows.append({
            'source_id': src_id, 'target_id': tgt_id, 'ships': int(round(send)),
            'angle': float(cand_angle[ci, 0].item()), 'eta': eta,
            'score_base_raw': score_base, 'score_margin_raw': score_base - roi_threshold,
            'features': feat,
        })
    return rows


## Data collection

In [ ]:
class CapturingTeacher:
    def __init__(self):
        self.events = []

    def reset(self):
        self.events.clear()
        try:
            C_MOD._RUNTIME.reset()
        except Exception:
            pass

    def __call__(self, obs, config=None):
        captured = {}
        original_greedy = C_MOD._greedy_select

        def wrapped_greedy(**kwargs):
            keep = {}
            for k, v in kwargs.items():
                if isinstance(v, torch.Tensor):
                    keep[k] = v.detach().cpu().clone()
                else:
                    keep[k] = v
            captured['kwargs'] = keep
            return original_greedy(**kwargs)

        C_MOD._greedy_select = wrapped_greedy
        try:
            moves = C_MOD.agent(obs)
        finally:
            C_MOD._greedy_select = original_greedy

        if moves and 'kwargs' in captured:
            player_id = int(obs.get('player', 0)) if isinstance(obs, dict) else int(obs.player)
            try:
                cand_rows = rows_from_greedy_capture(obs, player_id, captured['kwargs'])
            except Exception as e:
                print('[capture feature warn]', type(e).__name__, str(e)[:120])
                cand_rows = []
            self.events.append({
                'step': int(obs.get('step', 0)) if isinstance(obs, dict) else 0,
                'player_id': player_id,
                'moves': moves,
                'cand_rows': cand_rows,
            })
        return moves

def run_one_collection_game(opponent_path: Path, seed: int, teacher_slot: int, game_id: int):
    teacher = CapturingTeacher()
    teacher.reset()
    players = [teacher, str(opponent_path)] if teacher_slot == 0 else [str(opponent_path), teacher]
    env = make('orbit_wars', debug=False, configuration={'randomSeed': int(seed)})
    try:
        env.run(players)
    except Exception as e:
        return [], {'game_id': game_id, 'error': repr(e), 'opponent': str(opponent_path), 'seed': seed, 'slot': teacher_slot}

    rows = []
    for event in teacher.events:
        step_idx = int(event['step'])
        player_id = int(event['player_id'])
        cand_rows = event['cand_rows']
        for move_idx, mv in enumerate(event['moves']):
            if len(mv) < 3:
                continue
            sid, _, ships = int(mv[0]), float(mv[1]), int(mv[2])
            matched = match_action_to_candidate(mv, cand_rows)
            if matched is None:
                continue
            target_id = int(matched['target_id'])
            bad, parts = label_bad_move(
                env.steps, slot=teacher_slot, player_id=player_id,
                source_id=sid, target_id=target_id, step_idx=step_idx, eta=float(matched['eta']),
            )
            rows.append({
                'features': matched['features'],
                'label_bad': bad,
                'game_id': game_id,
                'step': step_idx,
                'move_idx': move_idx,
                'seed': int(seed),
                'teacher_slot': int(teacher_slot),
                'opponent': agent_label_for_path(opponent_path),
                'source_id': sid,
                'target_id': target_id,
                'ships': ships,
                'eta': float(matched['eta']),
                'score_base_raw': float(matched['score_base_raw']),
                'score_margin_raw': float(matched['score_margin_raw']),
                **parts,
            })
    info = {
        'game_id': game_id, 'error': None, 'opponent': agent_label_for_path(opponent_path),
        'seed': int(seed), 'slot': int(teacher_slot), 'rows': len(rows), 'steps': len(env.steps),
        'events': len(teacher.events),
    }
    return rows, info

def build_jobs():
    jobs = []
    gid = 0
    for opp in OPPONENT_PATHS:
        seeds = SELFPLAY_SEEDS if Path(opp).resolve() == TEACHER_PATH.resolve() else SEEDS
        for seed in seeds:
            for slot in (0, 1):
                gid += 1
                jobs.append((Path(opp), int(seed), int(slot), gid))
    return jobs

jobs = build_jobs()
print('collection games:', len(jobs))


collection games: 200


In [23]:
# Optional one-game smoke before full collection.
SMOKE = True
if SMOKE:
    rows, info = run_one_collection_game(OPPONENT_PATHS[0], 999, 0, 0)
    print('smoke info:', info)
    print('smoke rows:', len(rows))
    if rows:
        print('first row:', {k: rows[0][k] for k in rows[0] if k != 'features'})
        print('feature_dim:', len(rows[0]['features']))


smoke info: {'game_id': 0, 'error': None, 'opponent': 'submission_h', 'seed': 999, 'slot': 0, 'rows': 122, 'steps': 147, 'events': 121}
smoke rows: 122
first row: {'label_bad': 0, 'game_id': 0, 'step': 1, 'move_idx': 0, 'seed': 999, 'teacher_slot': 0, 'opponent': 'submission_h', 'source_id': 24, 'target_id': 0, 'ships': 10, 'eta': 5.0, 'score_base_raw': 46.0, 'score_margin_raw': 44.5, 'arrival_turn': 6, 'captured': 1, 'source_lost': 0, 'target_lost_quick': 0}
feature_dim: 20


In [24]:
# Full data collection. Sequential by default for reliability on Kaggle.
all_rows = []
infos = []
t0 = time.time()

for i, (opp, seed, slot, gid) in enumerate(jobs, start=1):
    rows, info = run_one_collection_game(opp, seed, slot, gid)
    all_rows.extend(rows)
    infos.append(info)
    status = 'ERR' if info.get('error') else 'ok'
    print(f'[{i:03d}/{len(jobs)}] {status} opp={agent_label_for_path(opp)} seed={seed} slot={slot} rows={len(rows)} total={len(all_rows)} t={time.time()-t0:.1f}s', flush=True)

assert all_rows, 'No rows collected. Check baseline/opponent paths or action matching.'
X = np.stack([r['features'] for r in all_rows]).astype(np.float32)
y = np.asarray([r['label_bad'] for r in all_rows], dtype=np.float32)
meta_game = np.asarray([r['game_id'] for r in all_rows], dtype=np.int32)

print('X:', X.shape, 'bad_rate:', float(y.mean()))

npz_path = OUT_DIR / 'submission_c_validator_dataset.npz'
np.savez_compressed(npz_path, features=X, labels_bad=y.astype(np.int64), meta_game=meta_game, feature_names=np.asarray(FEATURE_NAMES))
print('saved:', npz_path)

meta_path = OUT_DIR / 'submission_c_validator_rows.csv'
fieldnames = [k for k in all_rows[0].keys() if k != 'features']
with meta_path.open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in all_rows:
        w.writerow({k: v for k, v in r.items() if k != 'features'})
print('saved:', meta_path)

if pd is not None:
    df = pd.DataFrame([{k: v for k, v in r.items() if k != 'features'} for r in all_rows])
    display(df.groupby('opponent')['label_bad'].agg(['count', 'mean']).sort_values('count', ascending=False))


[001/200] ok opp=submission_h seed=101 slot=0 rows=35 total=35 t=9.2s
[002/200] ok opp=submission_h seed=101 slot=1 rows=93 total=128 t=24.3s
[003/200] ok opp=submission_h seed=102 slot=0 rows=65 total=193 t=36.3s
[004/200] ok opp=submission_h seed=102 slot=1 rows=40 total=233 t=49.5s
[005/200] ok opp=submission_h seed=103 slot=0 rows=52 total=285 t=61.1s
[006/200] ok opp=submission_h seed=103 slot=1 rows=99 total=384 t=89.5s
[007/200] ok opp=submission_h seed=104 slot=0 rows=39 total=423 t=101.5s
[008/200] ok opp=submission_h seed=104 slot=1 rows=33 total=456 t=111.8s
[009/200] ok opp=submission_h seed=105 slot=0 rows=67 total=523 t=136.6s
[010/200] ok opp=submission_h seed=105 slot=1 rows=55 total=578 t=148.8s
[011/200] ok opp=submission_h seed=106 slot=0 rows=22 total=600 t=157.1s
[012/200] ok opp=submission_h seed=106 slot=1 rows=54 total=654 t=171.2s
[013/200] ok opp=submission_h seed=107 slot=0 rows=71 total=725 t=187.6s
[014/200] ok opp=submission_h seed=107 slot=1 rows=55 total

,count,mean
opponent,,
submission_c_self,3568,0.688341
buddy,2723,0.619170
ver12,2694,0.680401
multi_focus_hybrid_v1a_enemy_prod_bias_v2,2424,0.636551
submission_h,2194,0.661805


## Training

In [ ]:
import torch.nn as nn

rng = np.random.default_rng(42)
games = np.unique(meta_game)
rng.shuffle(games)
n_val = max(1, int(0.20 * len(games)))
val_games = set(games[:n_val].tolist())
val_mask = np.asarray([g in val_games for g in meta_game], dtype=bool)

X_train_raw, y_train = X[~val_mask], y[~val_mask]
X_val_raw, y_val = X[val_mask], y[val_mask]

mean = X_train_raw.mean(axis=0).astype(np.float32)
std = X_train_raw.std(axis=0).astype(np.float32)
std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
X_train = (X_train_raw - mean) / std
X_val = (X_val_raw - mean) / std

print('train:', X_train.shape, 'bad_rate:', float(y_train.mean()))
print('val:  ', X_val.shape, 'bad_rate:', float(y_val.mean()))

class BadMoveValidator(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def precision_at_top_fraction(probs, labels, frac=0.10):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    if len(probs) == 0:
        return float('nan')
    k = max(1, int(math.ceil(len(probs) * frac)))
    idx = np.argsort(-probs)[:k]
    return float(labels[idx].mean())

def simple_auc(probs, labels):
    labels = np.asarray(labels).astype(np.int32)
    probs = np.asarray(probs)
    pos = labels == 1
    neg = labels == 0
    if pos.sum() == 0 or neg.sum() == 0:
        return float('nan')
    order = np.argsort(probs)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(probs) + 1)
    return float((ranks[pos].sum() - pos.sum() * (pos.sum() + 1) / 2) / (pos.sum() * neg.sum()))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BadMoveValidator(FEATURE_DIM).to(device)

Xt = torch.from_numpy(X_train.astype(np.float32)).to(device)
yt = torch.from_numpy(y_train.astype(np.float32)).to(device)
Xv = torch.from_numpy(X_val.astype(np.float32)).to(device)
yv = torch.from_numpy(y_val.astype(np.float32)).to(device)

pr = float(max(y_train.mean(), 1e-6))
pos_weight = torch.tensor([(1.0 - pr) / pr], dtype=torch.float32, device=device)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 40
BATCH = 512
best_state = None
best_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xt), device=device)
    losses = []
    for start in range(0, len(perm), BATCH):
        b = perm[start:start+BATCH]
        logits = model(Xt[b])
        loss = crit(logits, yt[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(loss.detach().cpu()))

    model.eval()
    with torch.no_grad():
        val_logits = model(Xv)
        val_loss = float(crit(val_logits, yv).detach().cpu())
        val_probs = torch.sigmoid(val_logits).detach().cpu().numpy()
    auc = simple_auc(val_probs, y_val)
    p10 = precision_at_top_fraction(val_probs, y_val, 0.10)
    acc05 = float(((val_probs >= 0.5).astype(np.float32) == y_val).mean())
    if not math.isnan(auc) and auc > best_auc:
        best_auc = auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'epoch {epoch:02d} train_loss={np.mean(losses):.4f} val_loss={val_loss:.4f} auc={auc:.3f} acc05={acc05:.3f} p_bad@top10={p10:.3f}')

if best_state is not None:
    model.load_state_dict(best_state)
print('best_auc:', best_auc)


train: (11040, 20) bad_rate: 0.6550724506378174
val:   (2563, 20) bad_rate: 0.6781116127967834
epoch 01 train_loss=0.4631 val_loss=0.4234 auc=0.793 acc05=0.705 p_bad@top10=0.973
epoch 05 train_loss=0.3742 val_loss=0.3440 auc=0.813 acc05=0.770 p_bad@top10=0.992
epoch 10 train_loss=0.3606 val_loss=0.3362 auc=0.821 acc05=0.772 p_bad@top10=0.992
epoch 15 train_loss=0.3506 val_loss=0.3313 auc=0.830 acc05=0.755 p_bad@top10=0.996
epoch 20 train_loss=0.3440 val_loss=0.3292 auc=0.834 acc05=0.758 p_bad@top10=0.984
epoch 25 train_loss=0.3391 val_loss=0.3271 auc=0.837 acc05=0.758 p_bad@top10=0.984
epoch 30 train_loss=0.3364 val_loss=0.3284 auc=0.837 acc05=0.768 p_bad@top10=0.988
epoch 35 train_loss=0.3326 val_loss=0.3277 auc=0.837 acc05=0.761 p_bad@top10=0.988
epoch 40 train_loss=0.3304 val_loss=0.3276 auc=0.837 acc05=0.766 p_bad@top10=0.992
best_auc: 0.8388638978972696


In [ ]:
# Export NumPy weights + scaler + manifest
model.eval()
sd = model.state_dict()
weights = {
    'l0_w': sd['net.0.weight'].detach().cpu().numpy().astype(np.float32),
    'l0_b': sd['net.0.bias'].detach().cpu().numpy().astype(np.float32),
    'l1_w': sd['net.2.weight'].detach().cpu().numpy().astype(np.float32),
    'l1_b': sd['net.2.bias'].detach().cpu().numpy().astype(np.float32),
    'l2_w': sd['net.4.weight'].detach().cpu().numpy().astype(np.float32),
    'l2_b': sd['net.4.bias'].detach().cpu().numpy().astype(np.float32),
    'x_mean': mean.astype(np.float32),
    'x_std': std.astype(np.float32),
}
weights_path = OUT_DIR / 'submission_c_badmove_mlp_weights.npz'
np.savez(weights_path, **weights)

manifest = {
    'agent': 'submission_c',
    'teacher_path': str(TEACHER_PATH),
    'model': 'MLP 30 -> 64 -> 32 -> 1 sigmoid p_bad',
    'feature_names': FEATURE_NAMES,
    'feature_dim': FEATURE_DIM,
    'dataset_rows': int(len(X)),
    'bad_rate': float(y.mean()),
    'best_val_auc': float(best_auc),
    'opponents': [agent_label_for_path(p) for p in OPPONENT_PATHS],
    'seeds': SEEDS,
    'selfplay_seeds': SELFPLAY_SEEDS,
    'intended_use': 'risk penalty before _greedy_select; do not use topk1',
    'suggested_initial_gate': {
        'alpha': 0.25,
        'max_penalty': 0.50,
        'score_margin_window': [-0.5, 1.0],
        'attack_only': True,
    },
}
manifest_path = OUT_DIR / 'submission_c_badmove_mlp_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('saved weights:', weights_path)
print('saved manifest:', manifest_path)
print(json.dumps(manifest['suggested_initial_gate'], indent=2))


saved weights: /kaggle/working/nn_validator_submission_c/submission_c_badmove_mlp_weights.npz
saved manifest: /kaggle/working/nn_validator_submission_c/submission_c_badmove_mlp_manifest.json
{
  "alpha": 0.25,
  "max_penalty": 0.5,
  "score_margin_window": [
    -0.5,
    1.0
  ],
  "attack_only": true
}


In [27]:
# Package artifacts for easy download from Kaggle output.
tar_path = WORK / 'submission_c_nn_validator_artifacts.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    for p in OUT_DIR.iterdir():
        if p.is_file():
            tar.add(p, arcname=f'nn_validator_submission_c/{p.name}')
print('artifact tar:', tar_path, f'{tar_path.stat().st_size/1024:.1f} KB')
with tarfile.open(tar_path) as tar:
    for m in tar.getmembers():
        print(' ', m.name, m.size)


artifact tar: /kaggle/working/submission_c_nn_validator_artifacts.tar.gz 722.7 KB
  nn_validator_submission_c/submission_c_badmove_mlp_weights.npz 18576
  nn_validator_submission_c/submission_c_badmove_mlp_manifest.json 1803
  nn_validator_submission_c/submission_c_validator_rows.csv 913907
  nn_validator_submission_c/submission_c_validator_dataset.npz 560631
